In [ ]:
import pandas as pd
import numpy as np
import os
%pwd
os.chdir('../')
%pwd

'/media/om/volume2/MLOPS/The Ultimate MLOPS Course/youtube_project/MLOPS_youtube_CTA_Project'

In [ ]:
df= pd.read_csv("./data/raw/youtube_10000_videos.csv")
df.head()

,category,channel_id,channel_name,subscriber_count,channel_view_count,channel_video_count,video_id,video_title,published_at,duration_seconds,view_count,like_count,comment_count,description,video_url
0,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,MOlaKBJ1nDA,प्रश्नावली 5.2 Class 10 Maths | NCERT Class 10...,2026-07-18T01:52:16Z,3988.0,427,21,3,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=MOlaKBJ1nDA
1,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,uXRH_uqCOXI,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-19T02:22:39Z,3606.0,528,32,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=uXRH_uqCOXI
2,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,q4Q4dg6JDE0,प्रश्नावली 5.3 Class 10 Maths | NCERT Class 10...,2026-07-22T01:57:46Z,3985.0,520,23,2,📚 Class 10 Maths Chapter 5 – Arithmetic Progre...,https://www.youtube.com/watch?v=q4Q4dg6JDE0
3,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,UTrN2vvjau8,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-23T01:56:33Z,4655.0,566,27,0,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=UTrN2vvjau8
4,Education,UCBg_mociSFb4ECbu91_cXJA,ABHYAAS EDUCATION,167000,5169057,1711,22S8ptzHl-g,Class 10 Maths | निर्देशांक ज्यामिति (Coordina...,2026-07-24T02:38:43Z,2185.0,355,23,3,Class 10 Maths | Coordinate Geometry Part 01 |...,https://www.youtube.com/watch?v=22S8ptzHl-g


In [3]:
all_columns=df.columns

In [4]:
required_columns=['category','duration_seconds', 'view_count']
df1=df[required_columns]

In [5]:
df1["duration_seconds_log"]= np.log1p(df1["duration_seconds"])
df1

/tmp/ipykernel_21304/3844972159.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["duration_seconds_log"]= np.log1p(df1["duration_seconds"])


,category,duration_seconds,view_count,duration_seconds_log
0,Education,3988.0,427,8.291296
1,Education,3606.0,528,8.190632
2,Education,3985.0,520,8.290544
3,Education,4655.0,566,8.445912
4,Education,2185.0,355,7.689829
...,...,...,...,...
9950,Travel,3719.0,66816,8.221479
9951,Travel,4181.0,232027,8.338545
9952,Travel,3743.0,85656,8.227910
9953,Travel,22.0,147597,3.135494


In [6]:
numerical_columns=['duration_seconds_log','view_count']
categorical_columns=['category']

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
import numpy as np

numeric_features = [
    'duration_seconds_log',
]

categorical_features = ["category"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log_transform", FunctionTransformer(np.log1p)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=1000,
            random_state=42,
            n_jobs=-1
        )),
    ]
)

In [8]:
X=df1.drop(columns=["duration_seconds","view_count"])
X

,category,duration_seconds_log
0,Education,8.291296
1,Education,8.190632
2,Education,8.290544
3,Education,8.445912
4,Education,7.689829
...,...,...
9950,Travel,8.221479
9951,Travel,8.338545
9952,Travel,8.227910
9953,Travel,3.135494


In [9]:
y= np.log1p(df1["view_count"])
y

0        6.059123
1        6.270988
2        6.255750
3        6.340359
4        5.874931
          ...    
9950    11.109713
9951    12.354613
9952    11.358106
9953    11.902248
9954    12.598442
Name: view_count, Length: 9955, dtype: float64

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)
predictions

array([ 9.02944268,  7.90049299, 12.64169182, ..., 11.32685043,
        6.0406824 ,  9.15151326], shape=(1991,))

In [11]:
X_test["category"]

4084             Food
3145          Fitness
5487           Gaming
1253    Entertainment
5564           Gaming
            ...      
6624             News
3639          Fitness
8482       Technology
449         Education
9406           Travel
Name: category, Length: 1991, dtype: object

In [12]:
from sklearn.metrics import r2_score, mean_squared_error
y_pred= model_pipeline.predict(X_test)
r2= r2_score(y_test,y_pred)
r2

0.16283668236463245